# ch04 Bonus 04：滑动窗口注意力（SWA）

> 对照官方 `rasbt/LLMs-from-scratch` ch04/06_swa
> **参考真实模型**：Mistral / Gemma 2 / Qwen2（部分）

## 一句话

每个 token 只看**局部窗口**内的前 `window_size` 个 token，而非全部历史。注意力复杂度从 O(n²) 降到 O(n·w)，长序列推理更快、显存更省。

## 为什么需要 SWA

标准自注意力的每个 token 关注所有历史 token，序列一长，计算和 KV 缓存都爆炸。滑动窗口把关注范围限制在最近的 `w` 个 token 内：

- **计算复杂度**：O(n²·d) → O(n·w·d)，w 通常远小于 n
- **感受野**：单层只看 w 个 token，但 **堆叠 L 层后等效感受野 ≈ L×w**，仍能捕捉远距离依赖
- **显存**：KV 缓存只需保留最近 w 个 token，可滚动覆盖旧值

> Mistral-7B 用 window_size=4096，配合 32 层后等效感受野达 ~130K token。

## 核心改造

在标准因果掩码之上，**再叠加一个滑动窗口掩码**：屏蔽掉距离超过 `window_size` 的 token。

In [ ]:
import torch
import torch.nn as nn


class SlidingWindowAttention(nn.Module):
    """滑动窗口注意力：每个 token 只关注前 window_size 个 token（含自身）。"""

    def __init__(self, d_in, d_out, context_length, num_heads, window_size,
                 dropout=0.0, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.window_size = window_size
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        # 因果掩码（上三角）
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool(),
        )

    def forward(self, x):
        b, n, _ = x.shape
        q = self.W_query(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_key(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_value(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = q @ k.transpose(2, 3)
        # 因果掩码：屏蔽未来 token
        causal = self.mask.bool()[:n, :n]
        # 滑窗掩码：屏蔽距离 > window_size-1 的历史 token
        # attn_scores[i, j] 中 i 是行(query)，j 是列(key)；屏蔽 i - j > window_size-1
        idx = torch.arange(n, device=x.device)
        sw_mask = (idx.unsqueeze(1) - idx.unsqueeze(0)) > (self.window_size - 1)
        combined_mask = causal | sw_mask
        attn_scores.masked_fill_(combined_mask, -torch.inf)

        attn_weights = torch.softmax(attn_scores / self.head_dim ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        out = (attn_weights @ v).transpose(1, 2).contiguous().view(b, n, self.d_out)
        return self.out_proj(out)

## 2. 验证掩码：token 7 在 window=3 时应只看 [5,6,7]

In [ ]:
# 直接验证掩码逻辑
n, w = 8, 3
idx = torch.arange(n)
sw_mask = (idx.unsqueeze(1) - idx.unsqueeze(0)) > (w - 1)
causal = torch.triu(torch.ones(n, n), diagonal=1).bool()
combined = causal | sw_mask

print("window_size=3 时各 token 的可见范围：")
for i in range(n):
    visible = [j for j in range(n) if not combined[i][j]]
    print(f"  token {i} ← {visible}")

assert [j for j in range(n) if not combined[7][j]] == [5, 6, 7]

## 3. 运行 SWA 层，对比计算量

In [ ]:
torch.manual_seed(123)
batch, seq, dim, n_heads = 2, 64, 768, 12
x = torch.randn(batch, seq, dim)

# 全局注意力 vs 滑窗（window=8）
swa = SlidingWindowAttention(dim, dim, 1024, n_heads, window_size=8)
out = swa(x)
print(f"SWA 输出: {tuple(out.shape)}")
print(f"\n复杂度对比 (seq={seq})：")
print(f"  全局注意力: O(seq²)   = {seq**2} 次注意力计算")
print(f"  滑窗(w=8):  O(seq·w)  = {seq * 8} 次注意力计算  ← 省 {100*(1-8/seq):.0f}%")
print(f"\n💡 堆叠 L 层后等效感受野 ≈ L × window_size，能弥补单层局部性。")

---
> 📌 本 notebook 实现 SWA 并验证掩码与复杂度优势。
> Mistral/Gemma2 的完整实现见官方 `ch04/06_swa`。